
#  Imports bibliotheque et monter Drive


In [ ]:

import os
import shutil
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

DRIVE_BASE  = '/content/drive/MyDrive/projet_chiens'
OUTPUT_BASE = f'{DRIVE_BASE}/audio_v1'
RAW_DIR     = f'{OUTPUT_BASE}/raw'

os.makedirs(RAW_DIR, exist_ok=True)

print("✅ Drive monté !")
print(f"📁 Output : {OUTPUT_BASE}")

Mounted at /content/drive
✅ Drive monté !
📁 Output : /content/drive/MyDrive/projet_chiens/audio_v1


# Télécharger DS1 et DS2 depuis Kaggle

In [ ]:

!pip install kaggle -q

# Télécharger DS1
!kaggle datasets download -d mmoreaux/audio-cats-and-dogs \
    -p /content/datasets/ds1 --unzip -q

# Télécharger DS2
!kaggle datasets download -d shivarao100/dog-voice-emotion-dataset \
    -p /content/datasets/ds2 --unzip -q

print("✅ Datasets téléchargés !")

Dataset URL: https://www.kaggle.com/datasets/mmoreaux/audio-cats-and-dogs
License(s): CC-BY-SA-3.0
Dataset URL: https://www.kaggle.com/datasets/shivarao100/dog-voice-emotion-dataset
License(s): apache-2.0
✅ Datasets téléchargés !


In [ ]:
# ============================================================
# DIAGNOSTIC — Voir le contenu réel de chaque dossier
# ============================================================

print("📂 Contenu réel de DS1 :\n")
for root, dirs, files in os.walk('/content/datasets/ds1'):
    wavs = [f for f in files if f.lower().endswith('.wav')]
    if wavs:
        print(f"  📁 {root}")
        print(f"     → {len(wavs)} fichiers WAV")
        print(f"     → exemples : {wavs[:3]}")
        print()

print("\n📂 Contenu réel de DS2 :\n")
for root, dirs, files in os.walk('/content/datasets/ds2'):
    wavs = [f for f in files if f.lower().endswith('.wav')]
    if wavs:
        print(f"  📁 {root}")
        print(f"     → {len(wavs)} fichiers WAV")
        print(f"     → exemples : {wavs[:3]}")
        print()

📂 Contenu réel de DS1 :

  📁 /content/datasets/ds1/cats_dogs
     → 277 fichiers WAV
     → exemples : ['dog_barking_1.wav', 'dog_barking_2.wav', 'cat_44.wav']

  📁 /content/datasets/ds1/cats_dogs/test/cats
     → 39 fichiers WAV
     → exemples : ['cat_79.wav', 'cat_88.wav', 'cat_129.wav']

  📁 /content/datasets/ds1/cats_dogs/test/test
     → 28 fichiers WAV
     → exemples : ['dog_barking_44.wav', 'dog_barking_54.wav', 'dog_barking_7.wav']

  📁 /content/datasets/ds1/cats_dogs/train/dog
     → 85 fichiers WAV
     → exemples : ['dog_barking_1.wav', 'dog_barking_2.wav', 'dog_barking_98.wav']

  📁 /content/datasets/ds1/cats_dogs/train/cat
     → 125 fichiers WAV
     → exemples : ['cat_44.wav', 'cat_167.wav', 'cat_153.wav']


📂 Contenu réel de DS2 :

  📁 /content/datasets/ds2/dog_bark_test
     → 13 fichiers WAV
     → exemples : ['dog_41.wav', 'dog_39.wav', 'dog_38.wav']

  📁 /content/datasets/ds2/dog_bark_train
     → 33 fichiers WAV
     → exemples : ['dog_17.wav', 'dog_20.wav', 'dog

In [ ]:
# ============================================================
# CELLULE 4 — Collecter les fichiers par classe
# ============================================================

SOURCES = {
    'bark' : [
        '/content/datasets/ds1/cats_dogs/train/dog',
        '/content/datasets/ds1/cats_dogs/test/test',
        '/content/datasets/ds2/dog_bark_train',
        '/content/datasets/ds2/dog_bark_test',
    ],
    'growl' : [
        '/content/datasets/ds2/dog_growl_train',
        '/content/datasets/ds2/dog_growl_test',
    ],
    'grunt' : [
        '/content/datasets/ds2/dog_grunt_train',
        '/content/datasets/ds2/dog_grunt_test',
    ],
}

# Collecter tous les chemins
fichiers_par_classe = {}

print("📊 Collecte par classe :\n")
total = 0
for label, dossiers in SOURCES.items():
    chemins = []
    for d in dossiers:
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.lower().endswith('.wav'):
                    chemins.append(os.path.join(d, f))
    fichiers_par_classe[label] = chemins
    print(f"  ✅ {label:10s} → {len(chemins)} fichiers")
    total += len(chemins)

print(f"\n  TOTAL   : {total} fichiers")
print(f"  TARGET  : {max(len(v) for v in fichiers_par_classe.values())} (classe majoritaire = bark)")
TARGET = max(len(v) for v in fichiers_par_classe.values())

📊 Collecte par classe :

  ✅ bark       → 159 fichiers
  ✅ growl      → 33 fichiers
  ✅ grunt      → 34 fichiers

  TOTAL   : 226 fichiers
  TARGET  : 159 (classe majoritaire = bark)


# Nettoyage fichiers corrompus
Le son c'est une vibration dans l'air. Pour le stocker dans un ordinateur, on prend des "photos" de cette vibration des milliers de fois par seconde.

SR = 22050 signifie :
→ on prend 22050 mesures par seconde
→ c'est le standard pour l'audio musical  utilisé par librosa et YAMNet

DURATION = 4 (Durée du clip)
Chaque fichier WAV a une durée différente :
On veut que tous les fichiers aient la même durée pour que le modèle reçoive toujours la même taille d'entrée.

4 secondes → suffisant pour capturer
             un aboiement ou grognement
             → standard utilisé dans la recherche
dans le cas ou :
dog_growl.wav (1 seconde)
→ librosa.load(duration=4)
→ charge 1 seconde + padding de silence
→ toujours 4 secondes en sortie

In [ ]:
# ============================================================
# CELLULE 5 — Nettoyage fichiers corrompus
# ============================================================
!pip install librosa -q
import librosa

SR       = 22050  # sample rate standard
DURATION = 4      # secondes par clip

print("🔍 Vérification des fichiers audio...\n")

total_supprimes = 0

for label, chemins in fichiers_par_classe.items():
    supprimes  = 0
    valides    = []

    for fpath in chemins:
        try:
            # Essayer de charger le fichier
            audio, sr = librosa.load(fpath, sr=SR,
                                     duration=DURATION)
            # Vérifier que le fichier n'est pas vide
            if len(audio) > SR * 0.5:  # minimum 0.5 secondes
                valides.append(fpath)
            else:
                supprimes += 1
        except:
            supprimes += 1

    fichiers_par_classe[label] = valides
    total_supprimes += supprimes
    status = "⚠️ " if supprimes > 0 else "✅"
    print(f"  {status} {label:10s} → {len(valides)} valides "
          f"| {supprimes} supprimés")

print(f"\n  Total supprimés : {total_supprimes}")
print("✅ Nettoyage terminé !")

🔍 Vérification des fichiers audio...

  ✅ bark       → 159 valides | 0 supprimés
  ✅ growl      → 33 valides | 0 supprimés
  ✅ grunt      → 34 valides | 0 supprimés

  Total supprimés : 0
✅ Nettoyage terminé !


# Sauvgarder sur drive

In [ ]:
# ============================================================
# CELLULE 6 — Copier les originaux sur Drive (bark/growl/grunt)
# ============================================================

print("💾 Copie des originaux sur Drive...\n")

for label, chemins in fichiers_par_classe.items():
    dest_dir = os.path.join(RAW_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    for fpath in chemins:
        fname = os.path.basename(fpath)
        dst   = os.path.join(dest_dir, fname)
        shutil.copy2(fpath, dst)

    print(f"  ✅ {label:10s} → {len(chemins)} fichiers copiés")

print(f"\n✅ Originaux sauvegardés dans : {RAW_DIR}")

💾 Copie des originaux sur Drive...

  ✅ bark       → 159 fichiers copiés
  ✅ growl      → 33 fichiers copiés
  ✅ grunt      → 34 fichiers copiés

✅ Originaux sauvegardés dans : /content/drive/MyDrive/projet_chiens/audio_v1/raw


J'ai collecté les données a partir de freesound vu qu'on a 3 classes c'est tres peu.

j'ai trouvé 2 autres classe whine et distress que j'ai cree qui contient 2 classes qui sont yelp( cri court de douleur) et howl (hurlement long  ) car les 2 sont des sons qui indiquent la douleur er le detresse est c'est un caractére anormal

In [ ]:
# ============================================================
# Vérification structure RAW_DIR complète
# ============================================================
print("📂 Contenu de RAW_DIR :\n")

total = 0
for dossier in sorted(os.listdir(RAW_DIR)):
    chemin = os.path.join(RAW_DIR, dossier)
    if os.path.isdir(chemin):
        n = len([f for f in os.listdir(chemin)
                 if f.lower().endswith('.wav')])
        print(f"  📁 {dossier:15s} → {n} fichiers")
        total += n

print(f"\n  TOTAL : {total} fichiers")
print(f"✅ Vérification terminée !")

📂 Contenu de RAW_DIR :

  📁 bark            → 159 fichiers
  📁 distress        → 38 fichiers
  📁 growl           → 33 fichiers
  📁 grunt           → 34 fichiers
  📁 whine           → 29 fichiers

  TOTAL : 293 fichiers
✅ Vérification terminée !


## Vérification durée avec MIN_dur et s'il ya un fichier  corrompus sur RAW_DIR

In [ ]:
# ============================================================
# CELLULE 7 — Vérification fichiers dans RAW_DIR
# ============================================================
import librosa

SR       = 22050
MIN_DUR  = 0.5   # minimum 0.5 secondes

print("🔍 Vérification des fichiers dans RAW_DIR...\n")

total_supprimes = 0

for dossier in sorted(os.listdir(RAW_DIR)):
    chemin = os.path.join(RAW_DIR, dossier)
    if not os.path.isdir(chemin):
        continue

    supprimes = 0
    valides   = 0

    for fname in os.listdir(chemin):
        if not fname.lower().endswith('.wav'):
            continue
        fpath = os.path.join(chemin, fname)
        try:
            audio, sr = librosa.load(fpath, sr=SR, duration=10)
            if len(audio) > SR * MIN_DUR:
                valides += 1
            else:
                os.remove(fpath)
                supprimes += 1
        except:
            os.remove(fpath)
            supprimes += 1

    total_supprimes += supprimes
    status = "⚠️ " if supprimes > 0 else "✅"
    print(f"  {status} {dossier:15s} → {valides} valides "
          f"| {supprimes} supprimés")

print(f"\n  Total supprimés : {total_supprimes}")
print("✅ Vérification terminée !")

🔍 Vérification des fichiers dans RAW_DIR...

  ✅ bark            → 159 valides | 0 supprimés
  ✅ distress        → 38 valides | 0 supprimés
  ✅ growl           → 33 valides | 0 supprimés
  ✅ grunt           → 34 valides | 0 supprimés
  ✅ whine           → 29 valides | 0 supprimés

  Total supprimés : 0
✅ Vérification terminée !


In [ ]:
# Supprimer AUG_DIR
import shutil
if os.path.exists(AUG_DIR):
    shutil.rmtree(AUG_DIR)
    print("✅ AUG_DIR supprimé")

✅ AUG_DIR supprimé


In [ ]:
# Vérification doublons RAW_DIR
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("🔍 Vérification doublons RAW_DIR...\n")

total_doublons = 0

for label in sorted(os.listdir(RAW_DIR)):
    src_dir = os.path.join(RAW_DIR, label)
    if not os.path.isdir(src_dir):
        continue

    all_files  = [f for f in os.listdir(src_dir)
                  if f.lower().endswith('.wav')]
    hashes_vus = {}
    doublons   = 0

    for fname in sorted(all_files):
        fpath = os.path.join(src_dir, fname)
        h     = md5_hash(fpath)
        if h in hashes_vus:
            os.remove(fpath)
            doublons += 1
        else:
            hashes_vus[h] = fname

    total_doublons += doublons

    # Recompter après suppression
    n_restants = len([f for f in os.listdir(src_dir)
                      if f.lower().endswith('.wav')])
    status = "⚠️ " if doublons > 0 else "✅"
    print(f"  {status} {label:15s} → {doublons} doublons "
          f"supprimés | {n_restants} restants")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Vérification terminée !")

🔍 Vérification doublons RAW_DIR...

  ⚠️  bark            → 1 doublons supprimés | 158 restants
  ⚠️  distress        → 1 doublons supprimés | 37 restants
  ✅ growl           → 0 doublons supprimés | 33 restants
  ✅ grunt           → 0 doublons supprimés | 34 restants
  ✅ whine           → 0 doublons supprimés | 29 restants

  Total doublons : 2
✅ Vérification terminée !


## Augmentation Classique
j'ai utilisé l'augmentation classique car notre dataset est trop petit pour entraîner un GAN audio comme WaveGAN (génère des sons WAV directement, pour des sons courts < 4s). Dans une version future avec plus de données, WaveGAN pourrait générer des sons plus variés et réalistes.

In [ ]:
# ============================================================
# CELLULE 8 — Augmentation audio jusqu'à TARGET=159
# ============================================================
import librosa
import numpy as np
import soundfile as sf
from itertools import combinations

TARGET   = 158
SR       = 22050
DURATION = 4
AUG_DIR  = f'{OUTPUT_BASE}/augmented'

os.makedirs(AUG_DIR, exist_ok=True)

def augment_audio_unique(audio, combo_id):
    """
    Chaque combo_id → transformation unique → pas de doublons
    """
    transformations = [
        lambda x: librosa.effects.time_stretch(x, rate=0.9),
        lambda x: librosa.effects.time_stretch(x, rate=1.1),
        lambda x: librosa.effects.pitch_shift(x, sr=SR, n_steps=2),
        lambda x: librosa.effects.pitch_shift(x, sr=SR, n_steps=-2),
        lambda x: x + 0.005 * np.random.randn(len(x)),  # bruit léger
        lambda x: np.flip(x),                            # reverse
        lambda x: librosa.effects.pitch_shift(x, sr=SR, n_steps=1),
        lambda x: librosa.effects.time_stretch(x, rate=0.8),
        lambda x: x * 1.2,                              # volume +
        lambda x: x * 0.8,                              # volume -
    ]

    combos = list(combinations(range(len(transformations)), 2))
    combo  = combos[combo_id % len(combos)]

    result = audio.copy()
    for i in combo:
        result = transformations[i](result)
    return result

def pad_or_trim(audio, sr, duration):
    """Normalise la durée à `duration` secondes"""
    target_len = sr * duration
    if len(audio) >= target_len:
        return audio[:target_len]
    else:
        return np.pad(audio, (0, target_len - len(audio)))

print(f"🎯 TARGET : {TARGET} fichiers par classe\n")
print("🔄 Augmentation en cours...\n")

for label in sorted(os.listdir(RAW_DIR)):
    src_dir  = os.path.join(RAW_DIR, label)
    dest_dir = os.path.join(AUG_DIR, label)
    os.makedirs(dest_dir, exist_ok=True)

    if not os.path.isdir(src_dir):
        continue

    # Collecter les originaux
    all_files = sorted([f for f in os.listdir(src_dir)
                        if f.lower().endswith('.wav')])
    n_orig = len(all_files)

    # Copier les originaux dans AUG_DIR
    for fname in all_files:
        src = os.path.join(src_dir, fname)
        dst = os.path.join(dest_dir, fname)
        audio, sr = librosa.load(src, sr=SR, duration=DURATION)
        audio     = pad_or_trim(audio, SR, DURATION)
        sf.write(dst, audio, SR)

    # Augmenter si nécessaire
    n_aug    = 0
    combo_id = 0
    if n_orig < TARGET:
        needed = TARGET - n_orig
        idx    = 0
        while n_aug < needed:
            fname = all_files[idx % len(all_files)]
            fpath = os.path.join(src_dir, fname)
            try:
                audio, sr = librosa.load(fpath, sr=SR,
                                         duration=DURATION)
                audio = pad_or_trim(audio, SR, DURATION)
                audio = augment_audio_unique(audio, combo_id)
                audio = pad_or_trim(audio, SR, DURATION)
                dst   = os.path.join(dest_dir,
                                     f'{label}_aug{n_aug:04d}.wav')
                sf.write(dst, audio, SR)
                n_aug    += 1
                combo_id += 1
            except Exception as e:
                pass
            idx += 1

    total  = n_orig + n_aug
    status = "✅" if n_aug == 0 else "🔧"
    print(f"  {status} {label:15s} {n_orig:3d} → {total:3d} "
          f"(+{n_aug} aug)")

print(f"\n✅ Augmentation terminée !")
print(f"📁 Résultat dans : {AUG_DIR}")

🎯 TARGET : 158 fichiers par classe

🔄 Augmentation en cours...

  ✅ bark            158 → 158 (+0 aug)
  🔧 distress         37 → 158 (+121 aug)
  🔧 growl            33 → 158 (+125 aug)
  🔧 grunt            34 → 158 (+124 aug)
  🔧 whine            29 → 158 (+129 aug)

✅ Augmentation terminée !
📁 Résultat dans : /content/drive/MyDrive/projet_chiens/audio_v1/augmented


In [ ]:
# ============================================================
# CELLULE 9 — Vérification doublons dans AUG_DIR
# ============================================================
import hashlib

def md5_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print("🔍 Vérification doublons dans AUG_DIR...\n")

total_doublons = 0

for label in sorted(os.listdir(AUG_DIR)):
    src_dir  = os.path.join(AUG_DIR, label)
    if not os.path.isdir(src_dir):
        continue

    all_files  = [f for f in os.listdir(src_dir)
                  if f.lower().endswith('.wav')]

    hashes_vus = {}
    doublons   = 0

    for fname in all_files:
        fpath = os.path.join(src_dir, fname)
        h     = md5_hash(fpath)
        if h in hashes_vus:
            doublons += 1
        else:
            hashes_vus[h] = fname

    total_doublons += doublons
    status = "⚠️ " if doublons > 0 else "✅"
    print(f"  {status} {label:15s} → {doublons} doublons")

print(f"\n  Total doublons : {total_doublons}")
print("✅ Vérification terminée !")

🔍 Vérification doublons dans AUG_DIR...

  ✅ bark            → 0 doublons
  ✅ distress        → 0 doublons
  ✅ growl           → 0 doublons
  ✅ grunt           → 0 doublons
  ✅ whine           → 0 doublons

  Total doublons : 0
✅ Vérification terminée !


In [ ]:
# Supprimer ancien FINAL_DIR
import shutil
if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)
    print("✅ Ancien FINAL_DIR supprimé")

✅ Ancien FINAL_DIR supprimé


In [ ]:
# ============================================================
# CELLULE 10 — Split 70/15/15
# ============================================================
import random

random.seed(42)

# Créer dossiers train/val/test
for split in ['train', 'val', 'test']:
    for label in os.listdir(AUG_DIR):
        if os.path.isdir(os.path.join(AUG_DIR, label)):
            os.makedirs(os.path.join(FINAL_DIR, split, label),
                        exist_ok=True)

print("📊 Split 70/15/15 en cours...\n")

for label in sorted(os.listdir(AUG_DIR)):
    src_dir = os.path.join(AUG_DIR, label)
    if not os.path.isdir(src_dir):
        continue

    all_files = sorted([f for f in os.listdir(src_dir)
                        if f.lower().endswith('.wav')])
    random.shuffle(all_files)

    n       = len(all_files)
    n_train = int(n * 0.70)  # 110 fichiers
    n_val   = int(n * 0.15)  #  23 fichiers

    splits = {
        'train' : all_files[:n_train],
        'val'   : all_files[n_train:n_train + n_val],
        'test'  : all_files[n_train + n_val:]
    }

    for split, files in splits.items():
        for fname in files:
            src = os.path.join(src_dir, fname)
            dst = os.path.join(FINAL_DIR, split, label, fname)
            shutil.copy2(src, dst)

    print(f"  ✅ {label:15s} → "
          f"train:{len(splits['train']):3d} | "
          f"val:{len(splits['val']):3d} | "
          f"test:{len(splits['test']):3d}")

print(f"\n✅ Split terminé !")
print(f"📁 Résultat dans : {FINAL_DIR}")

📊 Split 70/15/15 en cours...

  ✅ bark            → train:110 | val: 23 | test: 25
  ✅ distress        → train:110 | val: 23 | test: 25
  ✅ growl           → train:110 | val: 23 | test: 25
  ✅ grunt           → train:110 | val: 23 | test: 25
  ✅ whine           → train:110 | val: 23 | test: 25

✅ Split terminé !
📁 Résultat dans : /content/drive/MyDrive/projet_chiens/audio_v1/final


In [ ]:
# ============================================================
# CELLULE 11 — Génération labels.csv
# ============================================================
import pandas as pd

print("📝 Génération labels.csv...\n")

# Mapping label → groupe
label_to_groupe = {
    'bark'     : 'normal',
    'grunt'    : 'normal',
    'whine'    : 'normal',
    'growl'    : 'anormal',
    'distress' : 'anormal',
}

records = []

for split in ['train', 'val', 'test']:
    for label in sorted(os.listdir(os.path.join(FINAL_DIR, split))):
        label_dir = os.path.join(FINAL_DIR, split, label)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith('.wav'):
                records.append({
                    'filepath' : os.path.join(split, label, fname),
                    'label'    : label,
                    'groupe'   : label_to_groupe[label],
                    'split'    : split
                })

df = pd.DataFrame(records)
CSV_PATH = f'{OUTPUT_BASE}/labels.csv'
df.to_csv(CSV_PATH, index=False)

print(f"✅ labels.csv → {len(df)} lignes\n")
print("📊 Distribution par groupe :")
print(df.groupby(['split', 'groupe']).size().unstack(fill_value=0))
print("\n📊 Distribution par label :")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))
print(f"\n📁 Sauvegardé dans : {CSV_PATH}")

📝 Génération labels.csv...

✅ labels.csv → 790 lignes

📊 Distribution par groupe :
groupe  anormal  normal
split                  
test         50      75
train       220     330
val          46      69

📊 Distribution par label :
label  bark  distress  growl  grunt  whine
split                                     
test     25        25     25     25     25
train   110       110    110    110    110
val      23        23     23     23     23

📁 Sauvegardé dans : /content/drive/MyDrive/projet_chiens/audio_v1/labels.csv


In [ ]:
# ============================================================
# CELLULE 12 — ZIP final
# ============================================================
import shutil

ZIP_PATH = f'{OUTPUT_BASE}/audio_v1_final'

print("📦 Compression en cours...")

shutil.make_archive(
    ZIP_PATH,    # nom du zip
    'zip',       # format
    OUTPUT_BASE, # dossier parent
    'final'      # dossier à zipper
)

# Vérifier la taille
size = os.path.getsize(ZIP_PATH + '.zip')
print(f"✅ ZIP créé : {ZIP_PATH}.zip")
print(f"📁 Taille   : {size / (1024*1024):.1f} MB")

📦 Compression en cours...
✅ ZIP créé : /content/drive/MyDrive/projet_chiens/audio_v1/audio_v1_final.zip
📁 Taille   : 91.1 MB
